# La interfaz del agente

Los cuadernos anteriores construyeron un agente que consulta plazos, lee expedientes y busca en la normativa. Todos terminaban con una cadena de texto impresa en la salida de una celda, que es una forma muy cómoda de mentirse sobre si el sistema está terminado.

Este monta las dos formas de darle una cara, que corresponden a las dos direcciones del [capítulo](https://iraitzm.github.io/manual-ia-generativa/parts/agentes/interfaz.html):

1. **Un chat**, con Gradio. La interfaz vive fuera y llama al agente.
2. **MCP Apps**, donde el propio servidor MCP entrega la interfaz junto con la herramienta.

Y de paso la dirección que casi nadie espera: una interfaz de Gradio **exponiéndose ella misma como servidor MCP**, para que otro agente la use como herramienta.

## Preparación

In [ ]:
!pip install -q gradio fastmcp duckdb "transformers>=4.51" torch

In [ ]:
import pathlib
import subprocess
import sys

LOCAL = pathlib.Path("../../data/secretaria")
COLAB = pathlib.Path("manual-ia-generativa/data/secretaria")

if LOCAL.exists():
    base = LOCAL
else:
    if not COLAB.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--quiet",
             "https://github.com/IraitzM/manual-ia-generativa.git"],
            check=True,
        )
    base = COLAB

sys.path.insert(0, str(base.resolve()))

from secretaria import preparar

ctx = preparar()
con = ctx.conectar()

## El agente, en corto

Es el del [cuaderno del bucle](bucle-a-mano.ipynb), recortado a dos herramientas. Aquí no es el tema, es el motor.

In [ ]:
import json
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(ctx.modelo)
modelo = AutoModelForCausalLM.from_pretrained(ctx.modelo, dtype=torch.float32)
modelo.eval()

PATRON = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)
SISTEMA = ("Eres el asistente de la secretaría académica. Para responder debes llamar "
           "a una de las herramientas disponibles. No respondas de memoria.")
ALUMNO = "A2023001"


def consultar_plazo(tramite: str) -> str:
    filas = con.execute("""
        select tramite, fecha_inicio, fecha_fin from dim_plazo
        where tramite ilike '%' || ? || '%' order by fecha_inicio limit 3
    """, [tramite]).fetchall()
    if not filas:
        return f"No existe el trámite '{tramite}'."
    return "; ".join(f"{t}: del {i} al {f}" for t, i, f in filas)


def consultar_expediente(asignatura: str = "") -> str:
    filas = con.execute("""
        select s.asignatura, c.nota
        from fct_matriculas m
        join dim_asignatura s on s.asignatura_id = m.asignatura_id
        left join fct_calificaciones c on c.matricula_id = m.matricula_id
        where m.alumno_id = ? and (? = '' or lower(s.asignatura) like '%' || lower(?) || '%')
        order by s.asignatura limit 6
    """, [ALUMNO, asignatura, asignatura]).fetchall()
    if not filas:
        return "No estás matriculado de eso."
    return "; ".join(f"{a}: {n if n is not None else 'sin nota'}" for a, n in filas)


CATALOGO = {"consultar_plazo": consultar_plazo,
            "consultar_expediente": consultar_expediente}

ESQUEMAS = [
    {"type": "function", "function": {
        "name": "consultar_plazo",
        "description": "Fechas de inicio y fin de un trámite administrativo.",
        "parameters": {"type": "object", "properties": {
            "tramite": {"type": "string", "description": "beca, matricula, tfg, revision"}},
            "required": ["tramite"]}}},
    {"type": "function", "function": {
        "name": "consultar_expediente",
        "description": "Asignaturas y notas del alumno que pregunta.",
        "parameters": {"type": "object", "properties": {
            "asignatura": {"type": "string"}}, "required": []}}},
]


def agente(consulta, max_vueltas=3):
    mensajes = [{"role": "system", "content": SISTEMA},
                {"role": "user", "content": consulta}]
    for _ in range(max_vueltas):
        texto = tok.apply_chat_template(mensajes, tools=ESQUEMAS, tokenize=False,
                                        add_generation_prompt=True, enable_thinking=False)
        entrada = tok(texto, return_tensors="pt")
        with torch.no_grad():
            salida = modelo.generate(**entrada, max_new_tokens=100, do_sample=False,
                                     pad_token_id=tok.eos_token_id)
        bruto = tok.decode(salida[0][entrada.input_ids.shape[1]:],
                           skip_special_tokens=True).strip()
        encontrado = PATRON.search(bruto)
        if not encontrado:
            return bruto
        llamada = json.loads(encontrado.group(1))
        resultado = CATALOGO.get(llamada["name"], lambda **k: "herramienta desconocida")(
            **llamada.get("arguments", {}))
        mensajes.append({"role": "assistant", "content": "",
                         "tool_calls": [{"type": "function", "function": llamada}]})
        mensajes.append({"role": "tool", "name": llamada["name"], "content": resultado})
    return "(sin respuesta)"


print(agente("¿hasta cuándo puedo pedir la beca?"))

## Uno: el chat

[Gradio](https://www.gradio.app/) monta una interfaz de chat a partir de una función. La función recibe el mensaje y el historial, y devuelve la respuesta.

In [ ]:
import gradio as gr


def responder(mensaje, historial):
    """Lo único que Gradio necesita saber de nuestro agente."""
    return agente(mensaje)


chat = gr.ChatInterface(
    fn=responder,
    title="Secretaría académica",
    description="Consultas sobre plazos y expediente. Asistente automático.",
    examples=["¿hasta cuándo puedo pedir la beca?", "¿qué nota saqué en cálculo?"],
)

print("Construida. Todavía no hay servidor levantado.")

La interfaz no está lanzada: `chat` es solo un objeto. Conviene separar las dos cosas, porque la función se puede probar sin levantar nada, y eso es lo que permite tener pruebas automáticas de una interfaz.

In [ ]:
# Probar la interfaz sin interfaz.
print(responder("¿hasta cuándo puedo pedir la beca?", []))

In [ ]:
# Y ahora sí, el servidor. En Colab basta con chat.launch().
# Aquí lo lanzamos sin bloquear para que el cuaderno siga, y lo cerramos.
_, url, _ = chat.launch(prevent_thread_lock=True, quiet=True)
print("Sirviendo en", url)
chat.close()

Con eso ya hay algo que enseñarle a alguien, y es un salto enorme respecto a una celda de cuaderno. Fijaos en que la `description` dice que es un asistente automático: es el deber de transparencia del que habla [la normativa](https://iraitzm.github.io/manual-ia-generativa/parts/normativa/leyes.html), y aquí cuesta una línea.

Pero probad a pedirle algo irreversible.

## Lo que el chat no resuelve

Imaginad que el agente tuviera una herramienta para presentar solicitudes. La conversación sería algo así:

```
Alumno:    quiero pedir un certificado de notas
Asistente: ¿Confirmas que quieres presentar la solicitud?
Alumno:    sí
```

Ese "sí" es todo lo que separa a la universidad de registrar un trámite a nombre de una persona. Y es un "sí" que:

* No sabemos si se refiere a la solicitud o a otra cosa que el alumno tenía en la cabeza.
* No deja constancia de **qué** se le enseñó antes de aceptar.
* Se puede producir por inercia, después de tres mensajes seguidos donde "sí" era lo razonable.
* Lo interpreta un modelo, no un formulario.

El [capítulo del bucle]({B}/agentes/queesunagente.html#los-límites-no-son-un-detalle) decía que lo irreversible se confirma en código. Falta la otra mitad: se confirma **en la interfaz**, con el contenido delante. Y eso, en un chat, no se puede hacer con texto.

Aquí es donde entra la segunda parte.

## Dos: MCP Apps

La idea es que el servidor MCP no entregue solo el dato, sino también **la interfaz para verlo y actuar sobre él**. Son dos piezas.

Primero, un **recurso** con esquema `ui://`. Su `mimeType` tiene que ser exactamente `text/html;profile=mcp-app`, que es lo que le dice al cliente que eso es una aplicación y no un documento cualquiera.

In [ ]:
from fastmcp import Client, FastMCP

servidor = FastMCP("secretaria")

VISTA_PLAZOS = """<!doctype html>
<html lang="es">
<head><meta charset="utf-8"><style>
  body { font: 14px system-ui; margin: 0; padding: 12px; }
  table { border-collapse: collapse; width: 100%; }
  th, td { text-align: left; padding: 6px 8px; border-bottom: 1px solid #ddd; }
  th { color: #555; font-weight: 600; }
  button { margin-top: 10px; padding: 8px 14px; border: 0; border-radius: 6px;
           background: #3d7cb0; color: #fff; cursor: pointer; }
</style></head>
<body>
  <table id="tabla"><thead>
    <tr><th>Trámite</th><th>Desde</th><th>Hasta</th></tr>
  </thead><tbody></tbody></table>
  <button id="solicitar">Presentar solicitud</button>

<script>
  // El host empuja el resultado de la herramienta por postMessage.
  window.addEventListener("message", (evento) => {
    const m = evento.data;
    if (m.method !== "ui/notifications/tool-result") return;
    const cuerpo = document.querySelector("#tabla tbody");
    cuerpo.innerHTML = "";
    for (const fila of m.params.plazos) {
      const tr = document.createElement("tr");
      tr.innerHTML = `<td>${fila.tramite}</td><td>${fila.inicio}</td><td>${fila.fin}</td>`;
      cuerpo.appendChild(tr);
    }
  });

  // Y la vista le devuelve intenciones, nunca acciones ya hechas.
  document.querySelector("#solicitar").onclick = () => {
    window.parent.postMessage({
      jsonrpc: "2.0", id: 1, method: "tools/call",
      params: { name: "crear_solicitud", arguments: { tipo: "certificado_academico" } }
    }, "*");
  };

  window.parent.postMessage({ jsonrpc: "2.0", id: 0, method: "ui/initialize" }, "*");
</script>
</body></html>
"""


@servidor.resource(
    "ui://secretaria/plazos",
    mime_type="text/html;profile=mcp-app",
    meta={"ui": {"csp": {"connectDomains": [], "resourceDomains": []},
                 "prefersBorder": True}},
)
def vista_plazos() -> str:
    """Tabla de plazos con un botón para iniciar la solicitud."""
    return VISTA_PLAZOS

Y segundo, la **herramienta**, que es igual que cualquier otra salvo por una línea: `_meta.ui.resourceUri` apuntando al recurso de arriba.

In [ ]:
@servidor.tool(meta={"ui": {"resourceUri": "ui://secretaria/plazos"}})
def plazos_de(tramite: str) -> dict:
    """Fechas de inicio y fin de un trámite administrativo."""
    filas = con.execute("""
        select tramite, fecha_inicio, fecha_fin from dim_plazo
        where tramite ilike '%' || ? || '%' order by fecha_inicio limit 5
    """, [tramite]).fetchall()
    return {"plazos": [{"tramite": t, "inicio": str(i), "fin": str(f)} for t, i, f in filas]}


async with Client(servidor) as cliente:
    for h in await cliente.list_tools():
        print(f"herramienta {h.name}")
        print(f"  meta: {h.meta}")
    for r in await cliente.list_resources():
        print(f"recurso    {r.uri}")
        print(f"  mimeType: {r.mimeType}")

Ahí está el mecanismo entero. La herramienta lleva en sus metadatos la dirección de su interfaz, y el recurso se anuncia con el `mimeType` que lo identifica como aplicación.

Un cliente compatible, al ver esa herramienta, se descarga el recurso y lo mete en un `iframe` aislado antes incluso de llamarla. Nosotros podemos hacer lo mismo a mano.

In [ ]:
async with Client(servidor) as cliente:
    recurso = await cliente.read_resource("ui://secretaria/plazos")
    html = recurso[0].text
    datos = await cliente.call_tool("plazos_de", {"tramite": "beca"})

print(f"HTML recibido: {len(html)} caracteres")
print(f"Datos que el host empujaría a la vista:")
print(json.dumps(datos.data, ensure_ascii=False, indent=2))

### Verla de verdad

Un cuaderno no es un cliente MCP, así que el `iframe` con su aislamiento y su `postMessage` no lo vamos a tener. Pero el HTML es HTML, y se puede pintar aquí mismo con los datos ya dentro para ver qué recibiría el alumno.

In [ ]:
from IPython.display import HTML, display

# Simulamos lo que hace el host: entregarle a la vista el resultado de la herramienta.
# En un cliente real esto viaja como notificación ui/notifications/tool-result.
precargado = html.replace(
    'window.parent.postMessage({ jsonrpc: "2.0", id: 0, method: "ui/initialize" }, "*");',
    f'window.dispatchEvent(new MessageEvent("message", {{ data: '
    f'{{ method: "ui/notifications/tool-result", params: {json.dumps(datos.data)} }} }}));'
)

display(HTML(precargado))

Eso es lo que reemplaza a la frase "solicitud_beca_general: del 2026-08-01 al 2026-10-15" de los cuadernos anteriores. Y el botón es la diferencia de la que hablábamos: la confirmación tiene delante el dato concreto, no una conversación.

Fijaos en la línea del `onclick`. La vista **no crea la solicitud**: manda un `tools/call` al host pidiendo que la cree. Es la misma separación del [cuaderno del bucle](https://iraitzm.github.io/manual-ia-generativa/parts/agentes/queesunagente.html): quien ejecuta es el código de fuera, no quien lo pide. La vista es tan poco de fiar como el modelo, y por eso el protocolo la trata igual.

## La dirección contraria: la interfaz como servidor

Hay una vuelta de tuerca que sorprende la primera vez. Gradio puede exponer **la propia interfaz** como servidor MCP, convirtiendo cada función de la aplicación en una herramienta que otro agente puede llamar.

In [ ]:
def plazos_de_tramite(tramite: str) -> str:
    """Fechas de inicio y fin de un trámite administrativo de la secretaría."""
    return consultar_plazo(tramite)


panel = gr.Interface(fn=plazos_de_tramite, inputs="text", outputs="text",
                     title="Plazos de secretaría")

_, url, _ = panel.launch(prevent_thread_lock=True, quiet=True, mcp_server=True)
print("interfaz humana en", url)

async with Client(url.rstrip("/") + "/gradio_api/mcp/") as cliente:
    herramientas = await cliente.list_tools()
    for h in herramientas:
        print(f"  la misma función, como herramienta: {h.name}")
    respuesta = await cliente.call_tool(herramientas[0].name, {"tramite": "beca"})
    print("  llamada desde un agente:", respuesta.content[0].text)

panel.close()

La misma función, servida a la vez a una persona por HTTP y a un agente por MCP. No hay dos implementaciones ni dos contratos que mantener sincronizados.

Merece la pena pararse en lo que eso implica para el diseño. Si vuestra lógica de negocio está en funciones con tipos y un docstring decente, ya tenéis las dos interfaces. Si está enredada dentro de los manejadores de la interfaz, no tenéis ninguna de las dos, y ese es el argumento de siempre a favor de separar la lógica de la presentación, ahora con una razón nueva.

## Lo que hay que mirar con lupa

Recapitulando lo que hemos montado, hay dos sitios por donde entra código o contenido que no controlamos.

**El HTML del servidor MCP.** En este cuaderno lo hemos escrito nosotros. Si el servidor fuese de un tercero, ese HTML se ejecutaría en el navegador de vuestro usuario. El aislamiento del `iframe` es real y está bien pensado, y aun así la pregunta es la del capítulo: ¿le daríais a quien publicó ese servidor permiso para poner código en vuestra página?

**Lo que la vista mete en el contexto.** El método `ui/update-model-context` permite a la interfaz contarle al modelo lo que el usuario acaba de hacer. Es imprescindible para que la conversación siga teniendo sentido después de tocar un botón, y a la vez es un canal por el que una vista comprometida escribe directamente en el contexto del modelo.

Ninguna de las dos cosas es un motivo para no usar MCP Apps. Son los dos sitios donde mirar cuando llegue la revisión de seguridad, y conviene tenerlos localizados antes de que llegue.

## Ejercicios

**1. El formulario que falta.** La vista enseña plazos y tiene un botón. Añadidle lo que el capítulo pide para una confirmación: qué se va a presentar, a nombre de quién, y qué pasa después. Comparad el resultado con el "¿confirmas? sí" del chat.

**2. Probar la interfaz sin interfaz.** Escribid tres pruebas de `responder()` que se ejecuten sin levantar Gradio. Es lo que hace mantenible una interfaz, y casi nadie lo hace porque el framework invita a lo contrario.

**3. Streaming.** `agente()` tarda varios segundos y devuelve todo de golpe. Haced que `responder` sea un generador que va soltando el estado ("consultando plazos...", "redactando..."). Eso es, en pequeño, lo que estandariza AG-UI.

**4. La vista maliciosa.** Escribid una segunda vista que, en su `ui/update-model-context`, mande algo que intente cambiar el comportamiento del agente. No hace falta que funcione: el ejercicio es ver por dónde entra y qué haría falta para filtrarlo.

**5. Una herramienta con vista y otra sin ella.** Dad de alta `consultar_expediente` en el servidor MCP sin `_meta.ui`. Comparad las dos respuestas y decidid cuándo merece la pena una vista y cuándo el texto es mejor. La respuesta no es "siempre vista".

**6. La misma función, tres caras.** Coged `plazos_de_tramite` y servidla como interfaz de Gradio, como herramienta MCP y como función de Python en un cuaderno. Contad cuántas veces habéis escrito la lógica.

## Lo que os lleváis

* **El chat es la interfaz por defecto, no la buena.** Para un dato puntual sobra; para confirmar algo irreversible se queda muy corto.
* **La confirmación vive en la interfaz**, con el contenido delante. Un "sí" en una conversación no es una aprobación.
* **MCP Apps son dos líneas de protocolo**: un recurso `ui://` con `mimeType` `text/html;profile=mcp-app` y una herramienta que lo referencia con `_meta.ui.resourceUri`.
* **La vista pide, no hace.** Manda `tools/call` al host, igual que el modelo. Es la misma separación de siempre y por la misma razón.
* **Una función bien escrita ya es dos interfaces**, la humana y la de agente. Si no lo es, el problema estaba en la lógica, no en la interfaz.
* **Hay dos entradas que vigilar**: el HTML que sirve un tercero y lo que la vista escribe en el contexto del modelo.

Con esto se cierra la parte de construcción. Lo que viene es lo que hace falta para que esto sobreviva en una empresa, empezando por saber [qué ha pasado en cada petición](https://iraitzm.github.io/manual-ia-generativa/parts/produccion/observabilidad.html).